# Remaining position results validation

This notebook validates Nottingham Forest goalkeeper, midfield, winger and forward profiles against the supported Premier League templates. Defensive midfield is excluded because the supplied data does not contain two players with five eligible appearances.

Run `scripts/validate_forest_position_groups.py` first. Generated tables remain local because they derive from licensed event data.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_ROOT = PROJECT_ROOT / 'data' / 'processed'

GROUPS = {
    'goalkeeper': 'Goalkeeper',
    'central_midfield': 'Central midfield',
    'attacking_midfield': 'Attacking midfield',
    'wide_midfield': 'Wide midfield',
    'winger': 'Winger',
    'forward': 'Forward',
}
summary = pd.read_csv(PROCESSED_ROOT / 'forest_remaining_position_validation_summary.csv')
layout = pd.read_csv(PROCESSED_ROOT / 'remaining_position_layout_robustness.csv')
quality = pd.read_csv(PROCESSED_ROOT / 'remaining_position_validation_quality_checks.csv')

## 1. Reproducibility and quality checks

Bootstrap and zone-interval failure counts must be zero. Nominal-side observations are diagnostic rather than failures because midfielders and forwards can perform actions across the full width of the pitch. The coordinate transformation itself was already validated during template construction.

In [ ]:
quality

## 2. Qualified Forest overarching profiles

The table includes only players with at least five eligible appearances in the overarching position group. Similarity measures spatial conventionality relative to league peers, not performance quality.

In [ ]:
qualified = summary.loc[
    summary['template_eligible']
    & summary.apply(lambda row: row['template_group'] == row['position_group'], axis=1),
    ['position_group', 'player_name', 'layout', 'eligible_appearances', 'similarity_score'],
]
qualified.sort_values(['position_group', 'player_name', 'layout']).round(2)

## 3. Zone-layout robustness

The qualifying-player correlations test whether the 18- and 20-zone layouts produce similar scores and rankings. Both layouts should remain available because individual scores can still differ even when population-level agreement is high.

In [ ]:
layout.loc[
    layout['population'].eq('template-qualified players')
    & layout.apply(lambda row: row['template_group'] == row['position_group'], axis=1),
].round(3)

## 4. Split-half stability

Repeated split-half estimates are calculated only for profiles with at least ten eligible appearances. Smaller samples rely on bootstrap uncertainty and remain more tentative.

In [ ]:
split_tables = []
for stem, group in GROUPS.items():
    table = pd.read_csv(PROCESSED_ROOT / f'forest_{stem}_split_half_summary.csv')
    split_tables.append(table.loc[
        table['template_group'].eq(group) & table['split_half_status'].eq('calculated')
    ])
split_half = pd.concat(split_tables, ignore_index=True)
split_half[[
    'position_group', 'player_name', 'layout', 'appearance_count',
    'median_split_half_similarity', 'split_half_p05', 'split_half_p95'
]].round(2)

## 5. Action-family sensitivity

Pass and defensive-action profiles answer different football questions. Meaningful score differences confirm that later application views should not rely solely on the combined direct-action distribution.

In [ ]:
family_tables = []
for stem, group in GROUPS.items():
    table = pd.read_csv(PROCESSED_ROOT / f'forest_{stem}_action_family_sensitivity.csv')
    family_tables.append(table.loc[
        table['template_group'].eq(group) & table['family_template_eligible']
    ])
families = pd.concat(family_tables, ignore_index=True)
families.pivot_table(
    index=['position_group', 'player_name', 'layout'],
    columns='action_family',
    values='similarity_score',
).round(1)

## Decision

Goalkeeper, central-midfield, attacking-midfield and forward comparisons have qualifying Forest samples and can proceed toward the application. Wide-midfield and winger league templates are valid, but Forest samples remain provisional. Defensive midfield remains unavailable.

The two layouts show strong population-level agreement across the supported overarching groups, but both should remain visible. Separate passing and defensive-action views should also be retained. Nominal formation side must be interpreted as lineup context rather than a guarantee that every recorded action occurs on that side of the pitch.